# Phase 2: Baseline Model Training (TF-IDF + Logistic Regression)

In this phase, we build our local baseline classifier:
1. **TF-IDF Feature Extraction**: Unigrams and bigrams, sublinear term frequency scaling.
2. **Logistic Regression Classifier**: Balanced class weights, L2 regularization, L-BFGS solver.
3. **Comprehensive Evaluation**: Accuracy, Macro F1, Weighted F1, Precision, Recall, ROC-AUC, Confusion Matrix.
4. **Artifact Persistence**: Vectorizer, model, and end-to-end pipeline saved via `Joblib`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

from src.classification.model_loader import load_baseline_pipeline
from src.classification.predict import BaselineClassifier, predict_statement
from src.classification.train_baseline import train_baseline_model

print("Libraries loaded successfully!")

## 1. Train Baseline Model
Trains the TF-IDF + Logistic Regression pipeline on the official training set and evaluates on validation & test splits.

In [ ]:
metrics = train_baseline_model(
    data_dir=PROJECT_ROOT / "data" / "processed",
    output_dir=PROJECT_ROOT / "artifacts" / "models" / "baseline",
    max_features=10000,
    C=1.0
)

## 2. Visualize Test Set Confusion Matrix

In [ ]:
cm = np.array(metrics["test"]["confusion_matrix"])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real (0)", "Fake (1)"])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap="Blues", ax=ax, values_format="d")
plt.title("Baseline Test Confusion Matrix", fontsize=13, fontweight="bold")
plt.grid(False)
plt.tight_layout()
plt.show()

## 3. Interactive Local CPU Inference
Test real-time predictions without GPU.

In [ ]:
clf = BaselineClassifier(PROJECT_ROOT / "artifacts" / "models" / "baseline")

sample_claims = [
    "The unemployment rate fell to its lowest level in five decades.",
    "Secret documents prove the earth is surrounded by an ice wall guarded by the military.",
    "Congress approved a bipartisan infrastructure investment bill for bridges and roads.",
    "Drinking bleach cures common cold viruses within twenty four hours."
]

for claim in sample_claims:
    res = clf.predict(claim)
    print(f"Statement:  {res['statement']}")
    print(f"Verdict:    {res['label']} (Confidence: {res['confidence']*100:.1f}%)")
    print(f"P(Fake):    {res['prob_fake']:.4f} | P(Real): {res['prob_real']:.4f}")
    print("-" * 60)